# Preparing - Load library `aleph`

## Experiment configuration

In [ ]:
SEEDS = [11111, 22222, 33333, 44444, 55555]
DATASETS = [
    "QX_Lung",
    "QX_Marrow",
    "QS_Heart",
    "QS_Lung",
    "QS_Diaphragm",
    "QS_Limb_Muscle",
]

from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "topology_signature.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from inside the TopoCLR repository.")

BASE_DIR = PROJECT_ROOT / "data"  # Folder containing the local .h5 files.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MAX_TOPOLOGY_DIM = 1
TOPO_WEIGHT = 10
pretrain_epoch = 5
funetrain_epoch = 10

p_m = 0.3
k = 10
temperature = 0.7
dims = [256, 64, 32]
highly_genes = 1000
alpha = 1.0
beta = 0.1
gamma = 0.1
learning_rate = 0.0001
batch_size = 128
update_epoch = 50
noise_sd = 1.0
error = 0.001

RUN_ROBUSTNESS = False
ROBUSTNESS_DATASET = "QS_Diaphragm"  # Used only when RUN_ROBUSTNESS is True.

## Load library

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import scanpy as sc
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.autograd import Variable
import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import HeteroConv, SAGEConv
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from tqdm.notebook import tqdm
from sklearn.metrics.cluster import adjusted_rand_score
import random
from sklearn import metrics
import h5py
import scipy as sp
import scipy.sparse
import scanpy as sc

import sys

import tensorflow.compat.v1 as tf
tf.disable_eager_execution() # Critical for tf.placeholder and Session to work
from keras.layers import Dense, Activation, GaussianNoise

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 5000)
np.set_printoptions(threshold=np.inf)

from IPython.display import display

def load_expression(path):
    with h5py.File(path, "r") as data_file:
        counts = np.asarray(data_file["X"])
        labels = np.asarray(data_file["Y"]).reshape(-1)
    if counts.ndim != 2 or counts.shape[0] != labels.size or not labels.size:
        raise ValueError("Expected a count matrix and one label per cell.")
    return counts, labels

# 1. Module Code

In [ ]:
# --- UTILS ---
class dotdict(dict):
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__

def in_ipynb():
    try:
        shell = get_ipython().__class__.__name__
        if shell == "ZMQInteractiveShell": return True
        return False
    except NameError: return False

def smart_tqdm():
    if in_ipynb(): return tqdm.tqdm_notebook
    return tqdm.tqdm

def empty_safe(fn, dtype):
    def _fn(x):
        if x.size: return fn(x)
        return x.astype(dtype)
    return _fn

decode = empty_safe(np.vectorize(lambda _x: _x.decode("utf-8")), str)

# --- LOSS FUNCTIONS ---
MeanAct = lambda x: tf.clip_by_value(tf.exp(x), 1e-5, 1e6)
DispAct = lambda x: tf.clip_by_value(tf.nn.softplus(x), 1e-4, 1e4)

def _nan2zero(x): return tf.where(tf.is_nan(x), tf.zeros_like(x), x)
def _nan2inf(x): return tf.where(tf.is_nan(x), tf.zeros_like(x)+np.inf, x)

def _nelem(x):
    nelem = tf.reduce_sum(tf.cast(~tf.is_nan(x), tf.float32))
    return tf.cast(tf.where(tf.equal(nelem, 0.), 1., nelem), x.dtype)

def _reduce_mean(x):
    nelem = _nelem(x)
    x = _nan2zero(x)
    return tf.divide(tf.reduce_sum(x), nelem)

def neighbor_k_original(k, bank, z):
    result = tf.matmul(z, tf.transpose(bank))
    return tf.nn.top_k(result, k=k)[1]

def distance_k_mask(batchsize, k, bank, z, z_tilde):
    h_index = tf.reshape(neighbor_k_original(k, bank, z), [-1,1])
    h_index = tf.cast(h_index, dtype=tf.int32)
    line = tf.reshape(tf.repeat(np.arange(batchsize), k), [-1,1])
    line = tf.cast(line, dtype=tf.int32)
    index = tf.stack([line, h_index], 1)
    index = tf.reshape(index, [k*batchsize, 2])
    distance = tf.matmul(z_tilde, tf.transpose(bank))
    result = tf.gather_nd(distance, index)
    return tf.reshape(result, [-1, k])

def neighbor_k_loss(batchsize, k, bank, z, z_tilde, temperature):
    bank_normal = tf.nn.l2_normalize(bank, axis=1)
    z_normal = tf.nn.l2_normalize(z, axis=1)
    z_tilde_normal = tf.nn.l2_normalize(z_tilde, axis=1)
    numerator = tf.exp(distance_k_mask(batchsize, k, bank_normal, z_normal, z_tilde_normal) / temperature)
    numerator = tf.reduce_sum(numerator, 1) 
    denominator = tf.exp(tf.matmul(z_tilde_normal, tf.transpose(bank_normal)) / temperature) 
    denominator = tf.reduce_sum(denominator, 1) 
    loss = tf.log(numerator / denominator) 
    return -tf.reduce_sum(loss, [0]) / batchsize

def NB(theta, y_true, y_pred, mask=False, debug=False, mean=False):
    eps = 1e-10
    scale_factor = 1.0
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32) * scale_factor
    if mask:
        nelem = _nelem(y_true)
        y_true = _nan2zero(y_true)
    theta = tf.minimum(theta, 1e6)
    t1 = tf.lgamma(theta + eps) + tf.lgamma(y_true + 1.0) - tf.lgamma(y_true + theta + eps)
    t2 = (theta + y_true) * tf.log(1.0 + (y_pred / (theta + eps))) + (y_true * (tf.log(theta + eps) - tf.log(y_pred + eps)))
    final = _nan2inf(t1 + t2)
    if mean:
        if mask: final = tf.divide(tf.reduce_sum(final), nelem)
        else: final = tf.reduce_mean(final)
    return final

def ZINB(pi, theta, y_true, y_pred, ridge_lambda, mean=True, mask=False, debug=False):
    eps = 1e-10
    scale_factor = 1.0
    nb_case = NB(theta, y_true, y_pred, mean=False, debug=debug) - tf.log(1.0 - pi + eps)
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32) * scale_factor
    theta = tf.minimum(theta, 1e6)
    zero_nb = tf.pow(theta / (theta + y_pred + eps), theta)
    zero_case = -tf.log(pi + ((1.0 - pi) * zero_nb) + eps)
    result = tf.where(tf.less(y_true, 1e-8), zero_case, nb_case)
    ridge = ridge_lambda * tf.square(pi)
    result += ridge
    if mean:
        if mask: result = _reduce_mean(result)
        else: result = tf.reduce_mean(result)
    return _nan2inf(result)

def cal_dist(hidden, clusters):
    # Replaced K.sum, K.square, K.expand_dims, K.exp, K.transpose, and K.pow with tf equivalents
    dist1 = tf.reduce_sum(tf.square(tf.expand_dims(hidden, axis=1) - clusters), axis=2)
    temp_dist1 = dist1 - tf.reshape(tf.reduce_min(dist1, axis=1), [-1, 1])
    
    q = tf.exp(-temp_dist1)
    q = tf.transpose(tf.transpose(q) / tf.reduce_sum(q, axis=1))
    q = tf.pow(q, 2)
    q = tf.transpose(tf.transpose(q) / tf.reduce_sum(q, axis=1))
    
    dist2 = dist1 * q
    return dist1, dist2

In [ ]:
from sklearn.preprocessing import LabelEncoder
def read_clean(data):
    assert isinstance(data, np.ndarray)
    if data.dtype.type is np.bytes_:
        data = decode(data)
    if data.size == 1:
        data = data.flat[0]
    return data

def dict_from_group(group):
    assert isinstance(group, h5py.Group)
    d = dotdict()
    for key in group:
        if isinstance(group[key], h5py.Group):
            value = dict_from_group(group[key])
        else:
            value = read_clean(group[key][...])
        d[key] = value
    return d
        
def read_data(filename, sparsify=False, skip_exprs=False):
    with h5py.File(filename, "r") as f:
        obs = pd.DataFrame(dict_from_group(f["obs"]), index=decode(f["obs_names"][...]))
        var = pd.DataFrame(dict_from_group(f["var"]), index=decode(f["var_names"][...]))
        uns = dict_from_group(f["uns"])
        if not skip_exprs:
            exprs_handle = f["exprs"]
            if isinstance(exprs_handle, h5py.Group):
                mat = sp.sparse.csr_matrix((exprs_handle["data"][...], exprs_handle["indices"][...],
                                               exprs_handle["indptr"][...]), shape=exprs_handle["shape"][...])
            else:
                mat = exprs_handle[...].astype(np.float32)
                if sparsify:
                    mat = sp.sparse.csr_matrix(mat)
        else:
            mat = sp.sparse.csr_matrix((obs.shape[0], var.shape[0]))
    return mat, obs, var, uns

def prepro(data_path):
    # Modified to accept direct path instead of constructing it from dataname
    mat, obs, var, uns = read_data(data_path, sparsify=False, skip_exprs=False)
    if isinstance(mat, np.ndarray):
        X = np.array(mat)
    else:
        X = np.array(mat.toarray())
    cell_name = np.array(obs["cell_type1"])
    cell_type, cell_label = np.unique(cell_name, return_inverse=True)
    return X, cell_label

def normalize(adata, copy=True, highly_genes=None, filter_min_counts=True, size_factors=True, normalize_input=True, logtrans_input=True):
    if isinstance(adata, sc.AnnData):
        if copy: adata = adata.copy()
    else: raise NotImplementedError

    if filter_min_counts:
        sc.pp.filter_genes(adata, min_counts=1)
        sc.pp.filter_cells(adata, min_counts=1)
    
    adata.raw = adata.copy() if (size_factors or normalize_input or logtrans_input) else adata
    
    if size_factors:
        sc.pp.normalize_total(adata)
        # scNAME expects size_factors to be computed this way
        adata.obs['size_factors'] = adata.obs.n_counts / np.median(adata.obs.n_counts)
    else:
        adata.obs['size_factors'] = 1.0
        
    if logtrans_input: sc.pp.log1p(adata)
        
    if highly_genes != None:
        sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, n_top_genes=highly_genes, subset=True)
        
    if normalize_input: sc.pp.scale(adata)
        
    return adata

In [ ]:
import numpy as np
import tensorflow.compat.v1 as tf
from keras.layers import Dense, Activation, GaussianNoise
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# ==========================================
# 1. TOPOLOGICAL LOSS UTILITIES
# ==========================================
from topology_signature import TopologySignatureLoss

def tf_topological_loss(X_batch, Z_batch, max_dim=None):
    """Call the shared PyTorch loss with gradients in a TF v1 graph.

    CPU Python callback for notebook training; not standalone SavedModel export.
    """
    criterion = TopologySignatureLoss(
        max_dimension=MAX_TOPOLOGY_DIM if max_dim is None else max_dim,
        loss_type="MSE",
        distance_type="euclidean",
    )

    def value_and_gradients(x_np, z_np):
        with torch.enable_grad():
            x = torch.tensor(x_np, dtype=torch.float32, requires_grad=True)
            z = torch.tensor(z_np, dtype=torch.float32, requires_grad=True)
            value = criterion(x, z)
            if value.requires_grad:
                gx, gz = torch.autograd.grad(value, (x, z), allow_unused=True)
            else:
                gx = gz = None
            gx = torch.zeros_like(x) if gx is None else gx
            gz = torch.zeros_like(z) if gz is None else gz
            return (np.asarray(value.detach().cpu(), dtype=np.float32),
                    gx.detach().cpu().numpy(), gz.detach().cpu().numpy())

    @tf.custom_gradient
    def shared_loss(x, z):
        value, gx, gz = tf.py_func(
            value_and_gradients, [x, z], [tf.float32, tf.float32, tf.float32],
            stateful=True, name="shared_topology_signature_loss",
        )
        value.set_shape([])
        gx.set_shape(x.shape)
        gz.set_shape(z.shape)

        def gradient(upstream):
            return upstream * gx, upstream * gz

        return value, gradient

    return shared_loss(X_batch, Z_batch)

# ==========================================
# 2. scNAME NETWORK GENERATORS
# ==========================================
def mask_generator(p_m, x):
    return np.random.binomial(1, p_m, x.shape)

def pretext_generator(m, x):
    no, dim = x.shape
    x_bar = np.zeros([no, dim])
    for i in range(dim):
        idx = np.random.permutation(no)
        x_bar[:, i] = x[idx, i]
    x_tilde = x * (1 - m) + x_bar * m
    m_new = 1 * (x != x_tilde)
    return m_new, x_tilde

# ==========================================
# 3. AUTOENCODER CLASS
# ==========================================
class autoencoder(object):
    def __init__(self, dataname, n, batch_size, k, temperature, dims, cluster_num, alpha, beta, gamma, learning_rate, noise_sd=1.5, init='glorot_uniform', act='relu', use_topo=False, topo_weight=0.05):
        self.dataname = dataname
        self.n = n
        self.batch_size = batch_size
        self.k = k
        self.temperature = temperature
        self.dims = dims
        self.cluster_num = cluster_num
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.learning_rate = learning_rate
        self.noise_sd = noise_sd
        self.init = init
        self.act = act
        self.n_stacks = len(self.dims) - 1
        
        # New Topological Toggles
        self.use_topo = use_topo
        self.topo_weight = topo_weight
        
        self.sf_layer = tf.placeholder(dtype=tf.float32, shape=(None, 1))
        self.x_origin = tf.placeholder(dtype=tf.float32, shape=(None, self.dims[0]))
        self.x_count = tf.placeholder(dtype=tf.float32, shape=(None, self.dims[0]))
        self.x_tilde = tf.placeholder(dtype=tf.float32, shape=(None, self.dims[0]))
        self.batch = tf.placeholder(dtype=tf.int32, shape=(None, 1))
        self.m = tf.placeholder(dtype=tf.float32, shape=(None, self.dims[0]))
        
        self.bank = tf.get_variable(name=self.dataname + "/bank", dtype=tf.float32, initializer=tf.zeros([self.n, self.dims[-1]]), trainable=False)
        self.clusters = tf.get_variable(name=self.dataname + "/clusters_rep", shape=[self.cluster_num, self.dims[-1]], dtype=tf.float32, initializer=tf.glorot_uniform_initializer())

        def encoder(x_input, dims, init, act, n_stacks):
            h = x_input
            h = GaussianNoise(self.noise_sd, name='input_noise')(h)  
            for i in range(n_stacks - 1):
                h = Dense(units=dims[i + 1], kernel_initializer=init, name='encoder_%d' % i)(h) 
                h = GaussianNoise(self.noise_sd, name='noise_%d' % i)(h)
                h = Activation(act)(h) 
            latent = Dense(units=dims[-1], kernel_initializer=init, name='encoder_hidden')(h)
            h = latent
            for i in range(n_stacks - 1, 0, -1):
                h = Dense(units=dims[i], activation=act, kernel_initializer=init, name='decoder_%d' % i)(h)
            mask_estimator = Dense(units=dims[0], activation='sigmoid', kernel_initializer=init, name='mask_estimator')(h)
            pi = Dense(units=dims[0], activation='sigmoid', kernel_initializer=init, name='pi')(h)
            disp = Dense(units=dims[0], activation=DispAct, kernel_initializer=init, name='dispersion')(h)
            mean = Dense(units=dims[0], activation=MeanAct, kernel_initializer=init, name='mean')(h)
            return latent, mask_estimator, pi, disp, mean
       
        self.encoder_x_tilde, self.mask_estimator, _, _, _ = encoder(self.x_tilde, self.dims, self.init, self.act, self.n_stacks)
        self.encoder_x, _, self.pi, self.disp, self.mean = encoder(self.x_origin, self.dims, self.init, self.act, self.n_stacks)
        self.output = self.mean * tf.matmul(self.sf_layer, tf.ones((1, self.mean.get_shape()[1]), dtype=tf.float32))
        
        self.bank = tf.scatter_nd_update(self.bank, self.batch, self.encoder_x)
        self.likelihood_loss = ZINB(self.pi, self.disp, self.x_count, self.output, ridge_lambda=1.0)
        self.mask_loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(self.m, self.mask_estimator), 0)
        self.latent_idx, _ = cal_dist(self.bank, self.clusters)
        self.neighbor_loss = neighbor_k_loss(self.batch_size, self.k, self.bank, self.encoder_x, self.encoder_x_tilde, self.temperature)
        self.latent_dist1, self.latent_dist2 = cal_dist(self.encoder_x, self.clusters)
        self.kmeans_loss = tf.reduce_mean(tf.reduce_sum(self.latent_dist2, axis=1))
        
        # Base scNAME Loss
        self.base_loss = self.likelihood_loss + self.alpha*self.mask_loss + self.beta*self.neighbor_loss + self.gamma*self.kmeans_loss
        
        # Add Topological Loss if enabled
        if self.use_topo:
            self.topo_loss = tf_topological_loss(self.x_origin, self.encoder_x)
            self.total_loss = self.base_loss + (self.topo_weight * self.topo_loss)
        else:
            self.topo_loss = tf.constant(0.0) 
            self.total_loss = self.base_loss

        self.sess = None
        self.optimizer = tf.train.AdamOptimizer(self.learning_rate)
        self.pretrain_op = self.optimizer.minimize(self.likelihood_loss + self.alpha*self.mask_loss)
        self.train_op = self.optimizer.minimize(self.total_loss)

    def encode(self, X_data, size_factor=None, batch_size=256):
        """
        Encode input matrix X_data to latent embeddings and return as numpy ndarray.

        - X_data: numpy array of shape (n_samples, n_features) matching model input dims[0].
        - size_factor: optional numpy array of shape (n_samples, 1). If None, will use ones.
        - batch_size: inference batch size.

        Requires an active TensorFlow v1 Session stored as self.sess (created by pretrain/funetrain).
        """
        import numpy as _np

        if not hasattr(self, 'sess') or self.sess is None:
            raise RuntimeError("No active TensorFlow session found on the model. "
                               "Call pretrain(...) or keep the session open (do not call funetrain() which closes the session), "
                               "or modify funetrain() to not close the session before calling encode().")

        X = _np.asarray(X_data, dtype=_np.float32)
        n_samples = X.shape[0]

        if size_factor is None:
            sf = _np.ones((n_samples, 1), dtype=_np.float32)
        else:
            sf = _np.asarray(size_factor, dtype=_np.float32)
            if sf.shape[0] != n_samples:
                raise ValueError("size_factor must have same number of rows as X_data")

        embeddings = []
        for i in range(0, n_samples, batch_size):
            x_batch = X[i : i + batch_size]
            sf_batch = sf[i : i + batch_size]
            feed = {
                self.x_origin: x_batch,
                self.sf_layer: sf_batch
            }
            # Run encoder_x to get latent representation
            batch_latent = self.sess.run(self.encoder_x, feed_dict=feed)
            embeddings.append(batch_latent)

        return _np.vstack(embeddings)

    def pretrain(self, X, count_X, p_m, size_factor, batch_size, pretrain_epoch):
        print("Begin pretraining...")
        init = tf.group(tf.global_variables_initializer(), tf.local_variables_initializer())
        config_ = tf.ConfigProto()
        config_.gpu_options.allow_growth = True
        config_.allow_soft_placement = True
        self.sess = tf.Session(config=config_)
        self.sess.run(init)
        
        pre_index = 0
        for ite in range(pretrain_epoch):
            while True:
                if (pre_index + 1) * batch_size > X.shape[0]:
                    last_index = np.arange(pre_index * batch_size, (pre_index + 1) * batch_size) % len(X)
                    last_index_new = last_index.reshape([-1, 1])
                    x_batch = X[last_index]
                    m_batch = mask_generator(p_m, x_batch)
                    m_batch_new, x_tilde_batch = pretext_generator(m_batch, x_batch)

                    _, mask_loss, likelihood_loss, latent, bank = self.sess.run(
                        [self.pretrain_op, self.mask_loss, self.likelihood_loss, self.encoder_x, self.bank],
                        feed_dict={self.x_origin: x_batch, self.x_count: count_X[last_index], self.batch: last_index_new,
                                   self.m : m_batch_new, self.x_tilde: x_tilde_batch, self.sf_layer: size_factor[last_index]})
                    self.bank_current = bank
                    if ite % 10 == 0:
                       print(f'Epoch: {ite}/{pretrain_epoch}, Iteration: {pre_index}, Likelihood loss: {np.round(likelihood_loss, 4)}, Mask loss: {np.round(mask_loss, 4)}')
                    pre_index = 0
                    break
                else:
                    batch_index = np.arange(pre_index * batch_size, (pre_index + 1) * batch_size)
                    batch_index_new = batch_index.reshape([-1,1])
                    x_batch = X[batch_index]
                    m_batch = mask_generator(p_m, x_batch)
                    m_batch_new, x_tilde_batch = pretext_generator(m_batch, x_batch)
                    
                    _, mask_loss, likelihood_loss, latent, bank = self.sess.run(
                        [self.pretrain_op, self.mask_loss, self.likelihood_loss, self.encoder_x, self.bank],
                        feed_dict={self.x_origin: x_batch, self.x_count: count_X[batch_index], self.batch: batch_index_new,
                                   self.m : m_batch_new, self.x_tilde: x_tilde_batch, self.sf_layer: size_factor[batch_index]})
                    self.bank_current = bank              
                    pre_index += 1

    def funetrain(self, dataname, X, Y, count_X, p_m, size_factor, batch_size, funetrain_epoch, update_epoch, error):
        kmeans = KMeans(n_clusters=self.cluster_num, init="k-means++", n_init=10)
        self.kmeans_pred = kmeans.fit_predict(self.bank_current)
        self.last_pred = np.copy(self.kmeans_pred)
        self.sess.run(tf.assign(self.clusters, kmeans.cluster_centers_))
        self.kmeans_ARI = np.around(adjusted_rand_score(Y, self.kmeans_pred), 5)
        self.Y_pred_best = self.kmeans_pred
        print("Initial KMeans ARI:", self.kmeans_ARI)
        print("Begin fine-tuning...")

        fune_index = 0
        for i in range(1, funetrain_epoch + 1):
            while True:
                if (fune_index + 1) * batch_size > X.shape[0]:
                    last_index = np.arange(fune_index * batch_size, (fune_index + 1) * batch_size) % len(X)
                    last_index_new = last_index.reshape([-1, 1])
                    x_batch = X[last_index]
                    m_batch = mask_generator(p_m, x_batch)
                    m_batch_new, x_tilde_batch = pretext_generator(m_batch, x_batch)
                    
                    fetches = [self.train_op, self.base_loss, self.topo_loss, self.total_loss, self.latent_idx, self.encoder_x, self.bank]
                    feed = {self.x_origin: x_batch, self.x_count: count_X[last_index], self.batch: last_index_new,
                            self.m: m_batch_new, self.x_tilde: x_tilde_batch, self.sf_layer: size_factor[last_index]}
                    
                    _, base_l, topo_l, total_l, latent_idx_cur, latent, bank = self.sess.run(fetches, feed_dict=feed)
               
                    self.Y_pred = np.argmin(latent_idx_cur, axis=1)
              
                    if (i - 1) % 10 == 0:
                        self.ARI = np.around(adjusted_rand_score(Y, self.Y_pred), 5)
                        log_str = f"Epoch {i-1}/{funetrain_epoch} | Base Loss: {base_l:.4f}"
                        if self.use_topo:
                            log_str += f" | Topo Loss: {topo_l:.4f}"
                        log_str += f" | Total Loss: {total_l:.4f} | ARI: {self.ARI}"
                        print(log_str)
                        
                    fune_index = 0
                    break
                else:
                    batch_index = np.arange(fune_index * batch_size, (fune_index + 1) * batch_size)
                    batch_index_new = batch_index.reshape([-1, 1])
                    x_batch = X[batch_index]
                    m_batch = mask_generator(p_m, x_batch)
                    m_batch_new, x_tilde_batch = pretext_generator(m_batch, x_batch)
                    
                    fetches = [self.train_op, self.base_loss, self.topo_loss, self.total_loss, self.latent_idx, self.bank, self.encoder_x]
                    feed = {self.x_origin: x_batch, self.x_count: count_X[batch_index], self.batch: batch_index_new,
                            self.m: m_batch_new, self.x_tilde: x_tilde_batch, self.sf_layer: size_factor[batch_index]}
                    
                    _, base_l, topo_l, total_l, latent_idx_cur, bank, latent = self.sess.run(fetches, feed_dict=feed)
                  
                    self.Y_pred = np.argmin(latent_idx_cur, axis=1)
                    fune_index += 1
                    
            if i % update_epoch == 0:
                current_error = np.sum(self.Y_pred != self.last_pred) / len(self.last_pred)
                # print("current_error:", current_error)
                if current_error < error:
                    print(f"Early stopping at epoch {i} with error {current_error:.4f} < {error}")
                    break
                else:
                    self.last_pred = self.Y_pred
                    
        # self.sess.close()
        return self.Y_pred

# 2. Train Model

In [ ]:
def prepare_dataset(dataset):
    counts, labels = load_expression(BASE_DIR / f"{dataset}.h5")
    adata = sc.AnnData(np.ceil(counts).astype(np.float32))
    adata.obs["Group"] = LabelEncoder().fit_transform(labels)
    adata = normalize(adata, copy=True, highly_genes=highly_genes)
    features = adata.X.astype(np.float32)
    counts = np.asarray(adata.raw[:, adata.var_names].X, dtype=np.float32)
    labels = adata.obs["Group"].to_numpy()
    if len(labels) < 2:
        raise ValueError("At least two cells are required after preprocessing.")
    size_factors = adata.obs["size_factors"].to_numpy(dtype=np.float32).reshape(-1, 1)
    return dataset, features, counts, labels, size_factors


def train_and_evaluate(prepared, seed, weight, keep_model):
    dataset, features, counts, labels, size_factors = prepared
    order = np.random.permutation(len(labels))
    features, counts, labels, size_factors = (
        features[order], counts[order], labels[order], size_factors[order],
    )
    num_clusters = np.unique(labels).size
    with tf.Graph().as_default():
        tf.set_random_seed(seed)
        model = autoencoder(
            dataset, len(labels), batch_size, min(k, len(labels)), temperature,
            [features.shape[1], *dims], num_clusters, alpha, beta, gamma,
            learning_rate, noise_sd, use_topo=weight > 0, topo_weight=weight,
        )
        trained = False
        try:
            model.pretrain(features, counts, p_m, size_factors, batch_size, pretrain_epoch)
            predictions = model.funetrain(
                dataset, features, labels, counts, p_m, size_factors,
                batch_size, funetrain_epoch, update_epoch, error,
            )
            nmi = normalized_mutual_info_score(labels, predictions)
            ari = adjusted_rand_score(labels, predictions)
            trained = True
            return nmi, ari, model if keep_model else None
        finally:
            if model.sess is not None and not (trained and keep_model):
                model.sess.close()

In [ ]:
TRAINING_MODES = {"Without topology": 0, "With topology": TOPO_WEIGHT}

if RUN_ROBUSTNESS and ROBUSTNESS_DATASET not in DATASETS:
    raise ValueError("ROBUSTNESS_DATASET must be included in DATASETS.")
if pretrain_epoch < 1 or funetrain_epoch < 1 or TOPO_WEIGHT <= 0:
    raise ValueError("Use a positive topology loss weight and at least one epoch in both stages.")

def run_experiments():
    scores = []
    selected_pair = None
    for dataset in DATASETS:
        print(f"\nDataset: {dataset}", flush=True)
        try:
            prepared = prepare_dataset(dataset)
        except Exception as error:
            print(f"{dataset}: preparation failed ({type(error).__name__}: {error})", flush=True)
            continue

        for seed in SEEDS:
            keep_models = RUN_ROBUSTNESS and dataset == ROBUSTNESS_DATASET and selected_pair is None
            models = []
            for mode, weight in TRAINING_MODES.items():
                run_label = f"{dataset} | seed={seed} | {mode}"
                print(run_label, flush=True)
                try:
                    random.seed(seed)
                    np.random.seed(seed)
                    torch.manual_seed(seed)
                    nmi, ari, model = train_and_evaluate(prepared, seed, weight, keep_models)
                    if not np.isfinite([nmi, ari]).all():
                        raise FloatingPointError("Non-finite clustering score.")
                except Exception as error:
                    print(f"{run_label}: failed ({type(error).__name__}: {error})", flush=True)
                    continue

                scores.append((dataset, seed, mode, nmi, ari))
                if keep_models:
                    models.append(model)

            if keep_models and len(models) == len(TRAINING_MODES):
                selected_pair = (prepared, models)
                print(f"Using {dataset} | seed={seed} for robustness evaluation.", flush=True)
            elif keep_models:
                for model in models:
                    model.sess.close()

    results = pd.DataFrame(
        scores, columns=["Dataset", "Seed", "Mode", "NMI", "ARI"]
    ).astype({"NMI": "float64", "ARI": "float64"})
    return results, selected_pair


if globals().get("robustness_pair") is not None:
    for fitted_model in robustness_pair[1]:
        fitted_model.sess.close()

run_results, robustness_pair = run_experiments()

## NMI and ARI by dataset

Mean and sample standard deviation (SD, ddof=1) are shown when every configured seed succeeds for that dataset and mode.

In [ ]:
summary = run_results.groupby(["Dataset", "Mode"]).agg(
    completed_seeds=("Seed", "count"),
    NMI_mean=("NMI", "mean"),
    NMI_SD=("NMI", "std"),
    ARI_mean=("ARI", "mean"),
    ARI_SD=("ARI", "std"),
)
expected_rows = pd.MultiIndex.from_product(
    [DATASETS, TRAINING_MODES], names=["Dataset", "Mode"]
)
summary = summary.reindex(expected_rows).rename(columns={
    "completed_seeds": "Completed seeds",
    "NMI_mean": "NMI mean",
    "NMI_SD": "NMI SD",
    "ARI_mean": "ARI mean",
    "ARI_SD": "ARI SD",
})
summary["Completed seeds"] = summary["Completed seeds"].fillna(0).astype(int)

incomplete = summary["Completed seeds"] != len(SEEDS)
metric_columns = ["NMI mean", "NMI SD", "ARI mean", "ARI SD"]
summary.loc[incomplete, metric_columns] = np.nan
display(summary.reset_index().round(4))

## Optional robustness evaluation

Set RUN_ROBUSTNESS to True and choose a ROBUSTNESS_DATASET from DATASETS.
The robustness evaluation uses the first seed that completes both training modes for that dataset.

In [ ]:
if RUN_ROBUSTNESS:
    from utils.robustness_topology import (
        DEFAULT_LEVELS, DEFAULT_SEEDS, make_default_appliers, run_robustness_comparison,
    )
    from utils.robustness_clustering import run_clustering_robustness_comparison

    ROBUSTNESS_LEVELS = DEFAULT_LEVELS
    ROBUSTNESS_SEEDS = DEFAULT_SEEDS
    # For a quick smoke run: ROBUSTNESS_LEVELS = (0.0, 0.1); ROBUSTNESS_SEEDS = (33333,)
    ROBUSTNESS_OUTPUT_DIR = PROJECT_ROOT / "results" / "robustness"


    if robustness_pair is None:
        raise RuntimeError(f"No seed completed both modes for {ROBUSTNESS_DATASET}.")
    prepared, fitted_models = robustness_pair

    _, X, _, Y, size_factor = prepared
    model_1, model_2 = fitted_models

    def _tensorflow_encoder(model):
        def encode(x, sf):
            if model.sess is None or getattr(model.sess, "_closed", False):
                raise RuntimeError("Keep the trained scNAME TensorFlow session open for inference.")
            # Each model has its own graph after tf.reset_default_graph().
            with model.sess.graph.as_default(), model.sess.as_default():
                return model.encode(np.asarray(x, dtype=np.float32),
                                    size_factor=np.asarray(sf, dtype=np.float32))
        return encode

    pert_list = ["feature_dropout", "gaussian_noise"]
    ROBUSTNESS_APPLIERS = make_default_appliers(pert_list)
    models_for_robustness = {
        label: dict(get_representation_fn=_tensorflow_encoder(model),
                    X_clean=np.asarray(X, dtype=np.float32).copy(),
                    aux_clean=np.asarray(size_factor, dtype=np.float32).copy(),
                    perturbation_appliers=ROBUSTNESS_APPLIERS)
        for label, model in [("Without Topology", model_1), ("With Topology", model_2)]
    }
    robustness_labels = np.asarray(Y).reshape(-1)

    robustness_num_clusters = len(np.unique(robustness_labels))
    robustness_dataset = ROBUSTNESS_DATASET
    robustness_file_prefix = "scname" + "_" + Path(robustness_dataset).name

In [ ]:
if RUN_ROBUSTNESS:
    topology_results = run_robustness_comparison(
        models=models_for_robustness, dataset_name=robustness_dataset,
        perturbations=pert_list, levels=ROBUSTNESS_LEVELS, seeds=ROBUSTNESS_SEEDS,
        save_path=ROBUSTNESS_OUTPUT_DIR / (robustness_file_prefix + "_topology.csv"),
    )
    display(topology_results.head())

In [ ]:
if RUN_ROBUSTNESS:
    clustering_results = run_clustering_robustness_comparison(
        models=models_for_robustness, dataset_name=robustness_dataset,
        Y_true=robustness_labels, num_clusters=robustness_num_clusters,
        perturbations=pert_list, levels=ROBUSTNESS_LEVELS, seeds=ROBUSTNESS_SEEDS,
        save_path=ROBUSTNESS_OUTPUT_DIR / (robustness_file_prefix + "_clustering.csv"),
    )
    display(clustering_results.head())